[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/editorial-v2/notebooks/06_Equilibria_and_Stability.ipynb)

# DiveLab

## Notebook 06 — Equilibria and Stability

**From state to local behavior**

### Guiding question

If a neutrally buoyant diver is disturbed, does the diver return to the original depth or move away from it?

This notebook recovers the phase-plane investigation formerly attached to Notebook 03 and places it after the required state-space foundation. The model is educational and must not be used for dive planning or operational buoyancy decisions.

## Learning objectives

By the end of this notebook, you should be able to:

- verify an equilibrium of the nonlinear vertical model;
- calculate the local buoyancy sensitivity $k_B$;
- construct and interpret the Jacobian matrix;
- classify the equilibrium from its eigenvalues;
- identify stable and unstable eigendirections;
- read trajectory direction in a phase portrait;
- compare local linear predictions with nonlinear simulation;
- explain why quadratic drag does not appear in the first-order model at $v=0$.

## From Notebook 05 to Notebook 06

Notebook 05 implemented

$$
\dot{\mathbf{x}}=\mathbf{f}(\mathbf{x}),
\qquad
\mathbf{x}=
\begin{bmatrix}
z\\
v
\end{bmatrix}.
$$

An equilibrium satisfies $\mathbf{f}(\mathbf{x}^*)=\mathbf{0}$. Stability asks the more demanding question: what happens to trajectories that start close to $\mathbf{x}^*$?

## Model conventions and assumptions

- Depth $z\geq0$ increases downward.
- Velocity $v$ is positive upward, so $\dot z=-v$.
- Water density is constant.
- Boyle's law is isothermal and instantaneous.
- Fixed displaced volume is incompressible.
- Drag is quadratic: $F_D=-cv|v|$.
- The system is open loop and unforced.
- The analysis is local to one neutral operating state.

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

## Physical and model parameters

In [ ]:
rho = 1025.0          # seawater density [kg/m^3]
g = 9.80665           # gravitational acceleration [m/s^2]
p0 = 101_325.0        # surface absolute pressure [Pa]
mass = 85.0           # diver-and-equipment mass [kg]
surface_gas_volume = 8.0e-3  # flexible gas volume at surface [m^3]
drag_coefficient = 0.80
projected_area = 0.70  # [m^2]

drag_parameter = 0.5 * rho * drag_coefficient * projected_area

## Rebuild the nonlinear model

The physical relations are

$$
P_{\mathrm{abs}}(z)=P_0+\rho gz,
$$

$$
V_g(z)=V_{g0}\frac{P_0}{P_{\mathrm{abs}}(z)},
$$

$$
F_B(z)=\rho g\left[V_f+V_g(z)\right],
$$

and

$$
F_D(v)=-cv|v|.
$$

In [ ]:
def ambient_pressure(depth_m):
    # Absolute ambient pressure [Pa].
    return p0 + rho * g * depth_m


def gas_volume(depth_m):
    # Flexible gas volume [m^3].
    return surface_gas_volume * p0 / ambient_pressure(depth_m)


def buoyant_force(depth_m, fixed_volume_m3):
    # Upward buoyant force [N].
    return rho * g * (fixed_volume_m3 + gas_volume(depth_m))


def drag_force(velocity_m_s):
    # Signed drag force [N], positive upward.
    return -drag_parameter * velocity_m_s * abs(velocity_m_s)

## Construct and verify the equilibrium

Choose $V_f$ so the model is neutral at $z^*=20\ \mathrm{m}$:

$$
V_f=\frac{m}{\rho}-V_g(z^*).
$$

The complete equilibrium also requires $v^*=0$.

In [ ]:
equilibrium_depth = 20.0  # [m]
equilibrium_velocity = 0.0  # [m/s]
fixed_volume = mass / rho - gas_volume(equilibrium_depth)


def state_derivative(time_s, state):
    depth_m, velocity_m_s = state
    net_force = (
        buoyant_force(depth_m, fixed_volume)
        - mass * g
        + drag_force(velocity_m_s)
    )
    return np.array([-velocity_m_s, net_force / mass])


equilibrium_state = np.array([equilibrium_depth, equilibrium_velocity])
equilibrium_residual = state_derivative(0.0, equilibrium_state)

print(f"Fixed displaced volume: {fixed_volume * 1000:.3f} L")
print("Equilibrium state:", equilibrium_state)
print("Derivative residual:", equilibrium_residual)

The zero derivative confirms equilibrium in the ideal model. It does not yet establish stability.

## Inspect the force around equilibrium

At zero velocity, drag vanishes. The sign of

$$
F_B(z)-mg
$$

shows whether a small depth displacement produces upward or downward acceleration.

In [ ]:
depth_grid = np.linspace(equilibrium_depth - 2.0, equilibrium_depth + 2.0, 201)
buoyancy_imbalance = np.array([
    buoyant_force(z, fixed_volume) - mass * g
    for z in depth_grid
])

plt.figure(figsize=(8, 4.5))
plt.plot(depth_grid, buoyancy_imbalance)
plt.axhline(0.0, color="black", linewidth=1)
plt.axvline(equilibrium_depth, color="black", linestyle="--", linewidth=1)
plt.xlabel("Depth [m]")
plt.ylabel("Buoyancy imbalance [N]")
plt.title("Net upward force at zero velocity")
plt.grid(True)
plt.show()

## Interpretation

Shallower than $z^*$, the buoyancy imbalance is positive and accelerates the model upward. Deeper than $z^*$, the imbalance is negative and accelerates it downward. The force reinforces the displacement instead of restoring it.

## Local buoyancy sensitivity

Differentiate the buoyant-force relation:

$$
k_B
=
\left.\frac{dF_B}{dz}\right|_{z^*}
=
-\frac{(\rho g)^2V_{g0}P_0}
{\left(P_0+\rho gz^*\right)^2}.
$$

In [ ]:
def analytic_buoyancy_sensitivity(depth_m):
    denominator = p0 + rho * g * depth_m
    return -(rho * g) ** 2 * surface_gas_volume * p0 / denominator**2


def numerical_buoyancy_sensitivity(depth_m, step_m=1.0e-3):
    force_above = buoyant_force(depth_m - step_m, fixed_volume)
    force_below = buoyant_force(depth_m + step_m, fixed_volume)
    return (force_below - force_above) / (2.0 * step_m)


k_b = analytic_buoyancy_sensitivity(equilibrium_depth)
k_b_numerical = numerical_buoyancy_sensitivity(equilibrium_depth)

print(f"Analytic dF_B/dz:  {k_b:.6f} N/m")
print(f"Numerical dF_B/dz: {k_b_numerical:.6f} N/m")
print(f"Difference:         {k_b_numerical - k_b:+.3e} N/m")

The negative value is the physical source of the reinforcing mechanism. The close numerical agreement also checks the derivative used in the linearization.

## Linearize in perturbation coordinates

Define

$$
\boldsymbol{\xi}=
\begin{bmatrix}
\delta z\\
\delta v
\end{bmatrix}
=
\begin{bmatrix}
z-z^*\\
v-v^*
\end{bmatrix}.
$$

The local model is

$$
\dot{\boldsymbol{\xi}}=A\boldsymbol{\xi},
\qquad
A=
\begin{bmatrix}
0 & -1\\
k_B/m & 0
\end{bmatrix}.
$$

In [ ]:
jacobian = np.array([
    [0.0, -1.0],
    [k_b / mass, 0.0],
])

print("Jacobian A:")
print(jacobian)

## Eigenvalues and classification

Because $k_B<0$,

$$
\lambda_{\pm}=\pm\sqrt{-\frac{k_B}{m}}.
$$

One mode decays and the other grows.

In [ ]:
eigenvalues, eigenvectors = np.linalg.eig(jacobian)
stable_index = np.argmin(eigenvalues.real)
unstable_index = np.argmax(eigenvalues.real)

lambda_stable = float(eigenvalues[stable_index].real)
lambda_unstable = float(eigenvalues[unstable_index].real)

print("Eigenvalues [1/s]:", eigenvalues)
print(f"Stable eigenvalue:   {lambda_stable:+.6f} 1/s")
print(f"Unstable eigenvalue: {lambda_unstable:+.6f} 1/s")
print(f"Local growth timescale: {1.0 / lambda_unstable:.3f} s")

## Interpretation — a saddle point

The negative eigenvalue defines a stable eigendirection. Initial conditions exactly on that direction approach the equilibrium in the linear model.

The positive eigenvalue defines an unstable eigendirection. Any perturbation with a component in that direction eventually grows. Mixed-sign eigenvalues therefore classify the equilibrium as an **unstable saddle**.

Since neither eigenvalue has zero real part, the equilibrium is hyperbolic: the nonlinear model has the same local saddle geometry.

## Eigdirection slopes

For either eigenvalue,

$$
\delta v=-\lambda\,\delta z.
$$

The stable line has positive slope; the unstable line has negative slope in the $(\delta z,\delta v)$ plane.

In [ ]:
print(f"Stable slope   dv/dz = {-lambda_stable:+.6f} 1/s")
print(f"Unstable slope dv/dz = {-lambda_unstable:+.6f} 1/s")

## Nonlinear time histories after small perturbations

Start at the equilibrium depth with equal and opposite velocity perturbations. A stable equilibrium would keep both responses close and eventually return them to $(z^*,0)$.

In [ ]:
def simulate_nonlinear(initial_state, duration_s=25.0, sample_count=1001):
    sample_times = np.linspace(0.0, duration_s, sample_count)
    solution = solve_ivp(
        state_derivative,
        (0.0, duration_s),
        np.asarray(initial_state, dtype=float),
        t_eval=sample_times,
        rtol=1e-9,
        atol=1e-11,
    )
    if not solution.success:
        raise RuntimeError(solution.message)
    return solution


velocity_perturbations = {
    "upward perturbation": [equilibrium_depth, +0.05],
    "downward perturbation": [equilibrium_depth, -0.05],
}

perturbation_solutions = {
    label: simulate_nonlinear(state)
    for label, state in velocity_perturbations.items()
}

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

for label, solution in perturbation_solutions.items():
    axes[0].plot(solution.t, solution.y[0], label=label)
    axes[1].plot(solution.t, solution.y[1], label=label)

axes[0].axhline(equilibrium_depth, color="black", linestyle="--", linewidth=1)
axes[0].set_ylabel("Depth [m]")
axes[0].set_title("Divergence after opposite velocity perturbations")
axes[0].invert_yaxis()
axes[0].grid(True)
axes[0].legend()

axes[1].axhline(0.0, color="black", linestyle="--", linewidth=1)
axes[1].set_xlabel("Time [s]")
axes[1].set_ylabel("Upward velocity [m/s]")
axes[1].grid(True)
axes[1].legend()

plt.tight_layout()
plt.show()

## Reading the time histories

Drag initially opposes each velocity perturbation. Meanwhile, the state moves away from the equilibrium depth. The resulting change in gas volume changes buoyancy in the same direction as the displacement. The trajectories eventually depart from $(z^*,0)$.

Drag slows the departure but does not create a restoring force.

## First phase portrait

The phase plane removes time from the axes. Each point $(z,v)$ is one complete state. The same two responses can now be viewed as paths through state space.

In [ ]:
plt.figure(figsize=(8, 5.5))

for label, solution in perturbation_solutions.items():
    plt.plot(solution.y[0], solution.y[1], label=label)
    plt.scatter(solution.y[0, 0], solution.y[1, 0], marker="o", s=35)

plt.scatter(
    [equilibrium_depth],
    [equilibrium_velocity],
    color="black",
    marker="x",
    s=80,
    label="equilibrium",
)
plt.xlabel("Depth z [m]")
plt.ylabel("Upward velocity v [m/s]")
plt.title("Nonlinear trajectories in the phase plane")
plt.grid(True)
plt.legend()
plt.show()

## Reading the trajectory direction

- When $v>0$, $\dot z=-v<0$: the trajectory moves left, toward shallower depth.
- When $v<0$, $\dot z=-v>0$: the trajectory moves right, toward greater depth.

The initial markers identify where each trajectory starts. The vector-field plot below makes direction visible throughout the neighborhood.

## Nonlinear vector field and saddle eigendirections

At every state $(z,v)$, the vector $(\dot z,\dot v)$ gives the instantaneous direction of evolution. Arrow lengths are normalized here so direction remains visible; they do not represent derivative magnitude.

In [ ]:
z_values = np.linspace(equilibrium_depth - 1.2, equilibrium_depth + 1.2, 25)
v_values = np.linspace(-0.16, 0.16, 25)
Z, V = np.meshgrid(z_values, v_values)

dZ = -V
dV = np.empty_like(Z)
for row in range(Z.shape[0]):
    for column in range(Z.shape[1]):
        dV[row, column] = state_derivative(
            0.0,
            [Z[row, column], V[row, column]],
        )[1]

arrow_norm = np.hypot(dZ, dV)
arrow_norm[arrow_norm == 0.0] = 1.0
U = dZ / arrow_norm
W = dV / arrow_norm

depth_offsets = np.linspace(-1.2, 1.2, 200)
stable_line_velocity = -lambda_stable * depth_offsets
unstable_line_velocity = -lambda_unstable * depth_offsets

plt.figure(figsize=(9, 6))
plt.quiver(Z, V, U, W, color="0.55", alpha=0.75, pivot="mid")
plt.plot(
    equilibrium_depth + depth_offsets,
    stable_line_velocity,
    "--",
    linewidth=2,
    label="stable eigendirection",
)
plt.plot(
    equilibrium_depth + depth_offsets,
    unstable_line_velocity,
    "--",
    linewidth=2,
    label="unstable eigendirection",
)
plt.scatter([equilibrium_depth], [0.0], color="black", marker="x", s=90)
plt.xlabel("Depth z [m]")
plt.ylabel("Upward velocity v [m/s]")
plt.title("Local nonlinear vector field and saddle geometry")
plt.grid(True)
plt.legend()
plt.show()

## What the vector field shows

The stable eigendirection points toward the equilibrium in forward time. The unstable eigendirection points away from it. Most nearby states contain components in both directions; after the stable component decays, the unstable component dominates.

For the nonlinear system, the true stable and unstable manifolds are generally curved. The straight eigendirections are their tangents at the equilibrium.

## Linear trajectories along the eigendirections

The linear model isolates the two modal directions. Start with the same positive depth perturbation, once on the stable line and once on the unstable line.

In [ ]:
def linear_derivative(time_s, perturbation_state):
    return jacobian @ perturbation_state


def simulate_linear(initial_perturbation, duration_s=20.0):
    sample_times = np.linspace(0.0, duration_s, 501)
    return solve_ivp(
        linear_derivative,
        (0.0, duration_s),
        np.asarray(initial_perturbation, dtype=float),
        t_eval=sample_times,
        rtol=1e-10,
        atol=1e-12,
    )


depth_offset_0 = 0.40
stable_initial = np.array([
    depth_offset_0,
    -lambda_stable * depth_offset_0,
])
unstable_initial = np.array([
    depth_offset_0,
    -lambda_unstable * depth_offset_0,
])

stable_solution = simulate_linear(stable_initial)
unstable_solution = simulate_linear(unstable_initial)

In [ ]:
plt.figure(figsize=(8, 5.5))
plt.plot(
    equilibrium_depth + stable_solution.y[0],
    stable_solution.y[1],
    label="stable-direction trajectory",
)
plt.plot(
    equilibrium_depth + unstable_solution.y[0],
    unstable_solution.y[1],
    label="unstable-direction trajectory",
)
plt.scatter([equilibrium_depth], [0.0], color="black", marker="x", s=80)
plt.scatter(
    [equilibrium_depth + stable_initial[0], equilibrium_depth + unstable_initial[0]],
    [stable_initial[1], unstable_initial[1]],
    s=35,
    label="initial states",
)
plt.xlabel("Depth z [m]")
plt.ylabel("Upward velocity v [m/s]")
plt.title("Linear motion along the saddle eigendirections")
plt.grid(True)
plt.legend()
plt.show()

The stable-direction trajectory approaches the equilibrium. The unstable-direction trajectory departs from it. This does not make the equilibrium stable: an arbitrary perturbation almost never lies exactly on the stable line.

## Compare linear and nonlinear trajectories

Linearization should agree with the nonlinear model only near the operating state and over a limited time. Use the same small depth perturbation in both models.

In [ ]:
comparison_initial_perturbation = np.array([-0.10, 0.0])
comparison_duration = 15.0

linear_comparison = simulate_linear(
    comparison_initial_perturbation,
    duration_s=comparison_duration,
)
nonlinear_comparison = simulate_nonlinear(
    equilibrium_state + comparison_initial_perturbation,
    duration_s=comparison_duration,
    sample_count=501,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].plot(
    linear_comparison.t,
    linear_comparison.y[0],
    label="linear",
)
axes[0].plot(
    nonlinear_comparison.t,
    nonlinear_comparison.y[0] - equilibrium_depth,
    "--",
    label="nonlinear",
)
axes[0].set_xlabel("Time [s]")
axes[0].set_ylabel("Depth perturbation [m]")
axes[0].set_title("Depth perturbation")
axes[0].grid(True)
axes[0].legend()

axes[1].plot(
    equilibrium_depth + linear_comparison.y[0],
    linear_comparison.y[1],
    label="linear",
)
axes[1].plot(
    nonlinear_comparison.y[0],
    nonlinear_comparison.y[1],
    "--",
    label="nonlinear",
)
axes[1].scatter([equilibrium_depth], [0.0], color="black", marker="x")
axes[1].set_xlabel("Depth z [m]")
axes[1].set_ylabel("Upward velocity v [m/s]")
axes[1].set_title("Local phase trajectories")
axes[1].grid(True)
axes[1].legend()

plt.tight_layout()
plt.show()

## Interpretation

The linear and nonlinear responses initially agree because they share the same value and first derivative at the operating state. Differences grow as the trajectory reaches regions where reciprocal gas compression and quadratic drag can no longer be approximated by first-order terms.

## Why quadratic drag is absent from the Jacobian

For $F_D=-cv|v|$,

$$
\left.\frac{dF_D}{dv}\right|_{v=0}=0.
$$

A centered finite-difference estimate approaches zero as the velocity step shrinks.

In [ ]:
def numerical_drag_slope(step_m_s=1.0e-5):
    return (drag_force(step_m_s) - drag_force(-step_m_s)) / (2.0 * step_m_s)


for step in [1.0e-2, 1.0e-3, 1.0e-4, 1.0e-5]:
    print(
        f"step={step:.0e} m/s  "
        f"centered slope={numerical_drag_slope(step):+.6e} N s/m"
    )

print("The estimates converge to the exact derivative, zero.")

Quadratic drag remains physically important away from equilibrium. It changes the nonlinear trajectory and limits speed, but it does not remove the local saddle created by the negative buoyancy slope.

## Exercises

### 1. Change the equilibrium depth

Move the equilibrium to $15\ \mathrm{m}$. Recalculate $V_f$, $k_B$, the Jacobian and both eigenvalues. Compare the local growth timescale with the $20\ \mathrm{m}$ case.

In [ ]:
# Your code here

### 2. Change the flexible gas volume

Repeat the analysis for surface gas volumes of $4$, $8$, and $12\ \mathrm{L}$. Explain how the gas volume changes the buoyancy slope and eigenvalue magnitude.

In [ ]:
# Your code here

### 3. Read phase-plane directions

Choose four states, one in each quadrant around $(z^*,0)$. Evaluate $(\dot z,\dot v)$ and predict the initial direction before plotting it.

In [ ]:
# Your code here

### 4. Change quadratic drag

Compare nonlinear trajectories for three drag coefficients. Verify that their shapes change while the Jacobian at $v=0$ does not.

In [ ]:
# Your code here

## Challenge — map the validity of the linear model

Choose a quantitative error measure between linear and nonlinear trajectories. Repeat the comparison for increasing initial perturbation sizes and determine where the local approximation ceases to meet your chosen tolerance. State the time interval and tolerance explicitly.

## Summary

- Equilibrium requires both neutral buoyancy and zero vertical velocity.
- Flexible-gas buoyancy has negative depth sensitivity, $k_B<0$.
- The local Jacobian has eigenvalues $\lambda_{\pm}=\pm\sqrt{-k_B/m}$.
- Opposite-sign eigenvalues classify the operating state as an unstable saddle.
- The stable and unstable eigendirections organize the local phase portrait.
- Vector-field arrows reveal trajectory direction throughout the neighborhood.
- Linear and nonlinear trajectories agree locally but diverge as higher-order effects grow.
- Quadratic drag changes nonlinear motion but contributes no first-order term at $v=0$.

## Next notebook

Notebook 07 will examine observability and state estimation. The model state contains both depth and velocity, but a sensor may provide only noisy depth measurements. The next question is whether the missing state can be reconstructed from the measured output and the model.